# 🚀 Space Mission Analysis

## Exploring the History of Space Exploration

This project analyzes thousands of space missions launched from the beginning of the Space Race in 1957 through the dataset's latest available records.

The analysis focuses on understanding:

- Which organizations launched the most missions?
- How has the success rate of space missions changed over time?
- Which countries dominated space launches?
- How have launch costs evolved?
- How did the United States and Soviet Union compare during the Space Race?
- Which organizations and countries led space exploration in different periods?

### Dataset

The dataset contains historical space mission records scraped from Next Spaceflight and includes information about organizations, launch locations, dates, rocket status, launch prices, and mission outcomes.

### Tools & Technologies

- Python
- Pandas
- NumPy
- Matplotlib
- Seaborn
- Plotly
- ISO 3166 country codes

### Project Goals

The goal is to clean, explore, visualize, and interpret the dataset to uncover meaningful patterns in the history of space exploration.

### Import Statements

In [ ]:
import calendar
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from iso3166 import countries

ModuleNotFoundError: No module named 'iso3166'

### Notebook Presentation

In [ ]:
pd.options.display.float_format = '{:,.2f}'.format

## 📂 Load Dataset

In [ ]:
df_data = pd.read_csv("mission_launches.csv")

print(f"Dataset shape: {df_data.shape}")
df_data.head()

## 🔍 Preliminary Data Exploration
Before beginning the analysis, I first examine the structure, size, columns, missing values, and duplicate records in the dataset.

In [ ]:
print(df_data.shape)
print(df_data.columns)

In [ ]:
print(df_data.isna().sum())
df_data.duplicated().sum()

## 🧹 Data Cleaning

The dataset contains two unnamed columns that appear to be index artifacts rather than meaningful analytical variables. These columns are removed before continuing with the analysis.

The `Price` column also contains missing values, so its numeric format and missingness are handled carefully.

In [ ]:
# Remove unnecessary index columns
df_data = df_data.drop(columns=["Unnamed: 0.1", "Unnamed: 0"])

# Remove duplicate records
df_data = df_data.drop_duplicates()

# Convert Price to numeric
df_data["Price"] = pd.to_numeric(df_data["Price"], errors="coerce")

print(f"Dataset shape after cleaning: {df_data.shape}")
print("\nMissing values:")
print(df_data.isna().sum())

## 📅 Date Feature Engineering

In [ ]:
df_data["Date"] = pd.to_datetime(df_data["Date"], format="mixed")

df_data["Year"] = df_data["Date"].dt.year
df_data["Month"] = df_data["Date"].dt.month
df_data["Month_Name"] = df_data["Date"].dt.month_name()

The date column is converted to datetime format and additional year and month features are extracted for time-based analysis.

## 📊 Dataset Overview

After removing unnecessary columns and duplicate records, I inspect the data types and summary statistics of the cleaned dataset.

In [ ]:
df_data.info()

In [ ]:
df_data.describe(include="all").T

## 📈 Descriptive Statistics

In [ ]:
df_data.describe()

In [ ]:
df_data.describe(include="object")
df_data.isna().sum()

## 🚀 Rocket Status of Mission Records

The `Rocket_Status` column records whether the rocket associated with each mission was active or retired.

Here, I compare the number of mission records associated with each rocket status.

In [ ]:
rocket_status = (
    df_data["Rocket_Status"]
    .value_counts()
    .rename_axis("Rocket_Status")
    .reset_index(name="Missions")
)

rocket_status["Percentage"] = (
    rocket_status["Missions"]
    / rocket_status["Missions"].sum()
    * 100
)

rocket_status

In [ ]:
fig = px.bar(
    rocket_status,
    x="Rocket_Status",
    y="Missions",
    text="Percentage",
    title="Mission Records by Rocket Status",
    labels={
        "Rocket_Status": "Rocket Status",
        "Missions": "Number of Mission Records"
    }
)

fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside"
)

fig.update_layout(
    xaxis_title="Rocket Status",
    yaxis_title="Number of Mission Records",
    yaxis_range=[0, rocket_status["Missions"].max() * 1.15]
)

fig.show()

### Key Insight

The distribution shows how recorded missions are associated with active and retired rocket systems. Because the dataset contains mission records rather than a unique inventory of rocket models, these values should be interpreted as mission-record counts.

# Distribution of Mission Status

How many missions were successful?
How many missions failed?

In [ ]:
mission_status = (
    df_data["Mission_Status"]
    .value_counts()
    .rename_axis("Mission_Status")
    .reset_index(name="Missions")
)

mission_status["Percentage"] = (
    mission_status["Missions"]
    / mission_status["Missions"].sum()
    * 100
)

mission_status

In [ ]:
fig = px.bar(
    mission_status,
    x="Mission_Status",
    y="Missions",
    text="Percentage",
    title="Mission Outcomes",
    labels={
        "Mission_Status": "Mission Status",
        "Missions": "Number of Missions"
    }
)

fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside"
)

fig.update_layout(
    xaxis_title="Mission Status",
    yaxis_title="Number of Missions",
    yaxis_range=[0, mission_status["Missions"].max() * 1.15]
)

fig.show()

### Key Insight

Successful missions make up the largest share of the recorded missions, while complete failures represent a much smaller proportion. Partial and prelaunch failures account for relatively few records.

The percentage labels provide additional context alongside the raw mission counts.

## 💰 Launch Cost Analysis

The dataset does not contain launch prices for every mission. Therefore, cost-based analysis is limited to missions with a known `Price` value.

The available prices are used to understand the distribution and range of recorded launch costs.

In [ ]:
known_prices = df_data["Price"].dropna()

print(f"Missions with known launch prices: {len(known_prices)}")
print(f"Missions with missing launch prices: {df_data['Price'].isna().sum()}")
print(f"Percentage of missions with known prices: {df_data['Price'].notna().mean() * 100:.2f}%")

In [ ]:
fig = px.histogram(
    known_prices,
    nbins=30,
    title="Distribution of Recorded Launch Costs",
    labels={
        "value": "Launch Cost",
        "count": "Number of Missions"
    }
)

fig.update_layout(
    xaxis_title="Launch Cost",
    yaxis_title="Number of Missions"
)

fig.show()

In [ ]:
fig = px.box(
    known_prices,
    x=known_prices,
    title="Distribution and Outliers of Recorded Launch Costs",
    labels={
        "x": "Launch Cost"
    }
)

fig.update_layout(
    xaxis_title="Launch Cost"
)

fig.show()

### Key Insight

Recorded launch costs show substantial variation across missions, with some missions having considerably higher costs than the majority.

The histogram shows the overall distribution, while the box plot highlights the spread and potential high-cost outliers.

Because launch-price information is missing for many missions, these visualizations represent only missions with recorded launch costs.

## 🌍 Space Launches by Country

Launch locations contain both geographic information and country names. To compare countries consistently, the country component is extracted and standardized before calculating launch totals.

In [ ]:
df_data["Location"].head()
df_data["Country"] = df_data["Location"].str.split(",").str[-1].str.strip()
df_data["Country"] = df_data["Country"].replace({
    "Russia": "Russian Federation",
    "New Mexico": "USA",
    "Yellow Sea": "China",
    "Shahrud Missile Test Site": "Iran",
    "Pacific Missile Range Facility": "USA",
    "Barents Sea": "Russian Federation",
    "Gran Canaria": "USA"
})
df_data["Country"] = df_data["Country"].replace({
    "North Korea": "Korea, Democratic People's Republic of",
    "South Korea": "Korea, Republic of"
})
df_data["Country"] = df_data["Country"].replace({
    "Iran": "Iran, Islamic Republic of"
})
df_data["Country"] = df_data["Country"].replace({
    "Pacific Ocean": "Kiribati"
})
df_data["Country_Code"] = df_data["Country"].apply(
    lambda country: countries.get(country).alpha3
)
df_data[["Country","Country_Code"]].drop_duplicates()

In [ ]:
country_launches = (
    df_data["Country"]
    .value_counts()
    .rename_axis("Country")
    .reset_index(name="Launches")
)

country_launches.head(10)

In [ ]:
fig = px.choropleth(
    country_launches,
    locations="Country",
    locationmode="country names",
    color="Launches",
    hover_name="Country",
    hover_data={
        "Launches": True
    },
    title="Space Launches by Country",
    color_continuous_scale="Viridis"
)

fig.update_layout(
    geo=dict(
        showframe=False,
        showcoastlines=True
    ),
    margin=dict(l=0, r=0, t=60, b=0)
)

fig.show()

### Key Insight

Space-launch activity is geographically concentrated, with a relatively small number of countries accounting for a large share of recorded launches.

The map represents the location of the launch site rather than necessarily the nationality of the organization conducting the mission.

## ⚠️ Mission Failure Analysis
A mission is classified as a failure for this analysis when its recorded `Mission_Status` is anything other than `Success`.

This includes:

- Failure
- Partial Failure
- Prelaunch Failure

This definition allows the analysis to capture all missions that did not achieve a fully successful outcome.

In [ ]:
failure_statuses = [
    "Failure",
    "Partial Failure",
    "Prelaunch Failure"
]

failed_missions = df_data[
    df_data["Mission_Status"].isin(failure_statuses)
]

print(f"Total failed missions: {len(failed_missions)}")
print(f"Failure rate: {len(failed_missions) / len(df_data) * 100:.2f}%")

In [ ]:
failure_breakdown = (
    failed_missions["Mission_Status"]
    .value_counts()
    .rename_axis("Failure Type")
    .reset_index(name="Missions")
)

failure_breakdown

In [ ]:
plt.figure(figsize=(9, 5))

plt.bar(
    failure_breakdown["Failure Type"],
    failure_breakdown["Missions"]
)

plt.title("Breakdown of Non-Successful Missions")
plt.xlabel("Failure Type")
plt.ylabel("Number of Missions")
plt.xticks(rotation=20)

plt.tight_layout()
plt.show()

In [ ]:
failure_by_country = (
    failed_missions["Country"]
    .value_counts()
    .rename_axis("Country")
    .reset_index(name="Failed_Missions")
)

failure_by_country.head(10)

In [ ]:
fig = px.choropleth(
    failure_by_country,
    locations="Country",
    locationmode="country names",
    color="Failed_Missions",
    hover_name="Country",
    hover_data={
        "Failed_Missions": True
    },
    title="Unsuccessful Space Missions by Country",
    color_continuous_scale="Reds"
)

fig.update_layout(
    geo=dict(
        showframe=False,
        showcoastlines=True
    ),
    margin=dict(l=0, r=0, t=60, b=0)
)

fig.show()

### Key Insight

Unsuccessful missions are distributed unevenly across launch locations. Countries with greater historical launch activity can naturally have higher numbers of unsuccessful missions, so failure counts should be interpreted alongside overall launch volume.

# Create a Plotly Sunburst Chart of the countries, organisations, and mission status.

In [ ]:
fig = px.sunburst(
    df_data,
    path=["Country", "Organisation", "Mission_Status"],
    title="Space Missions by Country, Organization and Outcome"
)

fig.update_layout(
    margin=dict(t=60, l=0, r=0, b=0)
)

fig.show()

### Key Insight

The Sunburst chart provides a hierarchical view of the dataset, showing how missions are distributed across countries, organizations, and mission outcomes.

It allows multiple dimensions of the dataset to be explored within a single visualization and helps reveal which organizations and countries contribute most to the recorded mission activity.

## 🏆 Country Leadership Over Time
To understand how space-launch leadership changed over time, I compare countries based on both total launches and successful launches.

Comparing these two measures provides a more complete picture of space-launch dominance.

In [ ]:
df_data["Price"] = pd.to_numeric(df_data["Price"],errors="coerce")
total_spending = df_data.groupby("Organisation")["Price"].sum().sort_values(ascending=False)
top_10_spending = total_spending.head(10)

In [ ]:
country_yearly_launches = (
    df_data
    .groupby(["Year", "Country"])
    .size()
    .reset_index(name="Launches")
)

country_leaders = (
    country_yearly_launches
    .loc[
        country_yearly_launches.groupby("Year")["Launches"].idxmax()
    ]
    .sort_values("Year")
)

country_leaders.head(10)

In [ ]:
successful_missions = df_data[
    df_data["Mission_Status"] == "Success"
]

successful_country_yearly = (
    successful_missions
    .groupby(["Year", "Country"])
    .size()
    .reset_index(name="Successful_Launches")
)

successful_country_leaders = (
    successful_country_yearly
    .loc[
        successful_country_yearly
        .groupby("Year")["Successful_Launches"]
        .idxmax()
    ]
    .sort_values("Year")
)

successful_country_leaders.head(10)

In [ ]:
country_leadership_comparison = country_leaders[
    ["Year", "Country", "Launches"]
].merge(
    successful_country_leaders[
        ["Year", "Country", "Successful_Launches"]
    ],
    on="Year",
    suffixes=("_Total", "_Successful")
)

country_leadership_comparison.head(10)

In [ ]:
plt.figure(figsize=(10,6))
plt.barh(top_10_spending.index,top_10_spending.values)
plt.xlabel("Total Spending (Usd Millions)")
plt.ylabel("Organisations")
plt.title("Total Amount of Money Soent by Organisations")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### Key Insight

Comparing total launches with successful launches reveals whether the country leading in launch activity also led in successful missions.

In some years, the same country may lead both measures, while in other years the leadership can differ. This comparison provides a more meaningful view of space-launch performance than launch volume alone.

## 🏢 Organization Leadership Over Time

This analysis identifies the organization with the highest number of recorded launches in each year.

Tracking organizational leadership over time helps reveal how the space industry has evolved and which organizations have dominated launch activity during different periods.

In [ ]:
average_spending = df_data.groupby("Organisation")["Price"].mean().sort_values(ascending=False)
top_10_average = average_spending.head(10)

In [ ]:
organization_yearly_launches = (
    df_data
    .groupby(["Year", "Organisation"])
    .size()
    .reset_index(name="Launches")
)

organization_leaders = (
    organization_yearly_launches
    .loc[
        organization_yearly_launches.groupby("Year")["Launches"].idxmax()
    ]
    .sort_values("Year")
)

organization_leaders.head(10)

In [ ]:
fig = px.bar(
    organization_leaders,
    x="Year",
    y="Launches",
    color="Organisation",
    title="Leading Space Organization by Year",
    labels={
        "Year": "Year",
        "Launches": "Number of Launches",
        "Organisation": "Organization"
    }
)

fig.update_layout(
    xaxis_title="Year",
    yaxis_title="Launches",
    legend_title="Organization"
)

fig.show()

In [ ]:
plt.figure(figsize=(10,6))
plt.barh(top_10_average.index,top_10_average.values)
plt.xlabel("Average Cost Per Launch (Usd Millions)")
plt.ylabel("Organisations")
plt.title("Average Amount of Money Spent by Organisations per Launch")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### Key Insight

The organization leading space launches changes over time, reflecting shifts in launch capability, government investment, and the growth of commercial space companies.

This analysis highlights how space-launch activity has transitioned between different organizations across the history of the dataset.

## 📈 Launch Trends Over Time

This analysis examines how the number of space launches has changed over the years.

A rolling average is also used to smooth short-term fluctuations and make the broader long-term trend easier to identify.

In [ ]:
yearly_launches = (
    df_data
    .groupby("Year")
    .size()
    .reset_index(name="Launches")
)

yearly_launches.head()

In [ ]:
yearly_launches["Rolling_Average"] = (
    yearly_launches["Launches"]
    .rolling(window=5)
    .mean()
)

yearly_launches.head(10)

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=yearly_launches["Year"],
        y=yearly_launches["Launches"],
        mode="lines+markers",
        name="Annual Launches"
    )
)

fig.add_trace(
    go.Scatter(
        x=rolling_avg["Year"],
        y=rolling_avg["Rolling_Average"],
        mode="lines",
        name="5-Year Rolling Average"
    )
)

fig.update_layout(
    title="Global Space Launch Activity Over Time",
    xaxis_title="Year",
    yaxis_title="Number of Launches",
    hovermode="x unified",
    legend_title="Measure"
)

fig.show()

### Key Insight

Annual launch activity fluctuates considerably from year to year. The 5-year rolling average smooths these short-term fluctuations and makes the longer-term pattern in global launch activity easier to identify.

The comparison shows why a rolling average can be useful when analyzing historical time-series data with substantial year-to-year variation.

## 📅 Monthly Launch Patterns

This analysis examines how space launches are distributed across the months of the year.

Understanding monthly launch activity can reveal recurring seasonal patterns and periods of higher or lower launch frequency.

In [ ]:
monthly_launches = (
    df_data
    .groupby(["Month", "Month_Name"])
    .size()
    .reset_index(name="Launches")
    .sort_values("Month")
)

monthly_launches

In [ ]:
monthly_launches = df_data.resample("ME",on="Date").size()
monthly_launches.idxmax, monthly_launches.max()
rolling_avg = monthly_launches.rolling(12).mean()

In [ ]:
fig = px.bar(
    monthly_launches,
    x="Month_Name",
    y="Launches",
    title="Space Launches by Month",
    labels={
        "Month_Name": "Month",
        "Launches": "Number of Launches"
    }
)

fig.update_layout(
    xaxis_title="Month",
    yaxis_title="Number of Launches"
)

fig.show()

In [ ]:
plt.figure(figsize=(14,6))
plt.plot(
    monthly_launches.index,
    monthly_launches.values,
    label = "Monthly Launches"
)
plt.plot(
    rolling_avg.index,
    rolling_avg.values,
    label = "12-Month Rolling Average"
)
plt.xlabel("Date")
plt.ylabel("Number Of Launches")
plt.title("Number Of Launches On Month on Month")
plt.legend()
plt.grid()
plt.show()

### Key Insight

Launch activity varies across the months of the year rather than being evenly distributed.

The monthly distribution helps identify periods of higher and lower launch activity, although the observed pattern may also be influenced by historical changes in launch schedules, technology, and space-program priorities.

## 💰 Average Launch Cost Over Time

This analysis examines how the average recorded launch cost has changed over the years.

Because launch prices are missing for many missions, this analysis only includes missions with a recorded price.

In [ ]:
yearly_cost = (
    df_data
    .dropna(subset=["Price"])
    .groupby("Year")["Price"]
    .mean()
    .reset_index(name="Average_Price")
)

yearly_cost.head()

In [ ]:
fig = px.line(
    yearly_cost,
    x="Year",
    y="Average_Price",
    markers=True,
    title="Average Recorded Launch Cost Over Time",
    labels={
        "Year": "Year",
        "Average_Price": "Average Launch Cost"
    }
)

fig.update_layout(
    xaxis_title="Year",
    yaxis_title="Average Launch Cost"
)

fig.show()

In [ ]:
price_count_per_year = df_data.groupby(df_data["Date"].dt.year)["Price"].count()
avg_cost_by_year = df_data.groupby(df_data["Date"].dt.year)["Price"].mean()
avg_cost_by_year = avg_cost_by_year.dropna()

In [ ]:
fig = px.line(
    avg_cost_by_year,
    x="Year",
    y="Average_Cost",
    markers=True,
    title="Average Recorded Launch Cost Over Time",
    labels={
        "Year": "Year",
        "Average_Cost": "Average Launch Cost"
    }
)

fig.update_layout(
    xaxis_title="Year",
    yaxis_title="Average Launch Cost",
    hovermode="x unified"
)

fig.show()

### Key Insight

The average recorded launch cost varies across different periods of the dataset. Changes over time may reflect differences in launch vehicles, mission types, organizations, and the availability of recorded price information.

Because launch prices are missing for many missions, this analysis represents only missions with available cost data.

## 🏢 Top Organizations Over Time

This analysis tracks the organizations with the highest number of recorded launches across the years.

Examining organizations over time helps highlight changes in launch activity and the emergence or decline of major participants in the space industry.

In [ ]:
organization_yearly = (
    df_data
    .groupby(["Year", "Organisation"])
    .size()
    .reset_index(name="Launches")
)

yearly_leaders = (
    organization_yearly
    .loc[
        organization_yearly
        .groupby("Year")["Launches"]
        .idxmax()
    ]
    .sort_values("Year")
)

yearly_leaders.head()

In [ ]:
fig = px.scatter(
    yearly_leaders,
    x="Year",
    y="Launches",
    color="Organisation",
    hover_data=["Organisation", "Launches"],
    title="Leading Organization by Launches Each Year",
    labels={
        "Year": "Year",
        "Launches": "Number of Launches",
        "Organisation": "Leading Organization"
    }
)

fig.update_layout(
    xaxis_title="Year",
    yaxis_title="Number of Launches",
    hovermode="closest"
)

fig.show()

In [ ]:
leadership_summary = (
    yearly_leaders["Organisation"]
    .value_counts()
    .reset_index()
)

leadership_summary.columns = [
    "Organisation",
    "Years as Annual Leader"
]

leadership_summary

### Key Insight

The organization leading space-launch activity changes across different periods rather than remaining constant throughout the dataset.

Counting the number of years each organization appears as the annual leader provides an additional measure of long-term leadership beyond total launch volume.

# 🇺🇸🇷🇺 USA vs USSR: The Space Race

The Space Race was characterized by intense competition between the United States and the Soviet Union.

This section compares their recorded launch activity and mission success rates to explore how their space programs differed across the period represented in the dataset.

> **Note:** The USSR grouping includes the USSR and former Soviet launch locations represented in the dataset. Therefore, this comparison should be interpreted as a dataset-based comparison rather than a complete historical measure of the two nations' space programs.

## 📊 USA vs USSR: Year-by-Year Launch Activity

Comparing launch activity year by year provides a clearer view of how the space programs of the United States and Soviet Union changed throughout the Space Race.

Rather than relying only on overall totals, this analysis highlights periods when one side launched substantially more missions than the other.

In [ ]:
usa_ussr_yearly = (
    usa_ussr
    .groupby(["Year", "Group"])
    .size()
    .reset_index(name="Launches")
)

usa_ussr_yearly.head()

In [ ]:
fig = px.line(
    usa_ussr_yearly,
    x="Year",
    y="Launches",
    color="Group",
    markers=True,
    title="USA vs USSR / Former Soviet Republics: Launches Over Time",
    labels={
        "Year": "Year",
        "Launches": "Number of Launches",
        "Group": "Space Program"
    }
)

fig.update_layout(
    xaxis_title="Year",
    yaxis_title="Number of Launches",
    legend_title="Space Program"
)

fig.show()

In [ ]:
usa_ussr_pivot = (
    usa_ussr_yearly
    .pivot(
        index="Year",
        columns="Group",
        values="Launches"
    )
    .fillna(0)
)

usa_ussr_pivot["Difference"] = (
    usa_ussr_pivot["USSR / Former Soviet Republics"]
    - usa_ussr_pivot["USA"]
)

usa_ussr_pivot.head()

### Key Insight

The comparison demonstrates that launch volume and mission success rate provide different perspectives on space-program activity.

The year-by-year launch chart highlights changes in launch intensity over time, while the success-rate comparison provides context about the outcomes of those recorded missions.

Neither metric alone is sufficient to determine which space program was more successful overall. Historical achievements also depend on mission objectives, technological milestones, scientific contributions, and other factors that are not captured by these two measures.

## 🚀 USA vs USSR: Mission Success

Launch volume alone does not tell the complete story of space-program performance.

This analysis compares the number of successful missions associated with the USA and USSR / Former Soviet Republics to provide an additional perspective on their launch activity.

In [ ]:
successful_usa_ussr = usa_ussr[
    usa_ussr["Mission_Status"] == "Success"
]

successful_counts = (
    successful_usa_ussr["Group"]
    .value_counts()
    .reset_index()
)

successful_counts.columns = ["Group", "Successful_Launches"]

successful_counts

In [ ]:
success_rate_comparison = (
    usa_ussr
    .groupby("Group")["Mission_Status"]
    .apply(lambda x: (x == "Success").mean() * 100)
    .reset_index(name="Success_Rate")
)

success_rate_comparison

In [ ]:
fig = px.bar(
    success_rate_comparison,
    x="Group",
    y="Success_Rate",
    title="USA vs USSR / Former Soviet Republics: Mission Success Rate",
    labels={
        "Group": "Space Program",
        "Success_Rate": "Success Rate (%)"
    },
    text="Success_Rate"
)

fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside"
)

fig.update_layout(
    xaxis_title="Space Program",
    yaxis_title="Success Rate (%)",
    yaxis_range=[0, 100]
)

fig.show()

### Key Insight

Comparing success rates provides a different perspective from comparing launch volume alone.

A space program may conduct many launches while having a different proportion of successful missions. Therefore, launch count and success rate should be considered separately when evaluating historical launch activity.

# 🎯 Key Findings

The analysis of historical space missions reveals several important patterns:

### 🚀 1. Space launches are concentrated among major organizations

A relatively small number of organizations account for a large share of recorded launches, showing the historical concentration of space-launch capabilities among major space programs and companies.

### 📈 2. Launch activity has changed significantly over time

The yearly launch trend shows substantial variation across different periods. The rolling average helps reveal longer-term changes that are less obvious from individual yearly values.

### 🌍 3. Space-launch activity is geographically concentrated

A limited number of countries account for a large proportion of launch activity. However, launch location should not be interpreted as the nationality of the organization conducting the mission.

### 💰 4. Launch costs vary considerably

Recorded launch prices show significant variation. Since prices are missing for many missions, cost-related findings represent only the subset of missions with available price information.

### 🏢 5. Organizational leadership changes over time

The organizations responsible for the highest number of launches have changed across different periods, reflecting the evolution of the global space industry.

### 🇺🇸🇷🇺 6. The Space Race shows changing launch activity between the USA and USSR

Comparing the USA with the USSR and former Soviet launch locations reveals that launch activity varied substantially across the different periods of the Space Race.

### ✅ 7. Launch volume and mission success are different measures

The organization or country conducting the most launches is not necessarily the one with the highest success rate. Examining both measures provides a more complete view of historical launch performance.

# 🏁 Conclusion

This project explored historical space missions to understand how launch activity, mission outcomes, organizations, countries, and launch costs have changed over time.

Through data cleaning, exploratory analysis, statistical summaries, and interactive visualizations, the project highlights the evolution of the space-launch industry from the early Space Race to the modern era.

The analysis also demonstrates the importance of considering multiple measures—such as launch volume, success rate, geographic distribution, and cost—rather than relying on a single metric to understand the development of space exploration.